# DSPy Adapters & Evaluation — Formatting In, Judging Out

**Week 6 | Notebook 10 of 12**

**What you'll learn:**
- What an **Adapter** is: the layer that formats signatures into LM prompts and parses
  replies back into typed fields
- All 5 adapters from the [API reference](https://dspy.ai/current/api/): `Adapter`,
  `ChatAdapter`, `XMLAdapter`, `JSONAdapter`, `TwoStepAdapter` — and when each should be used
- The metric contract every DSPy judge follows
- All 6 evaluation tools: `answer_exact_match`, `answer_passage_match`, `SemanticF1`,
  `CompleteAndGrounded`, `Evaluate`, `EvaluationResult`
- How adapters and metrics plug into optimizers (Notebook 9) and RAG (Notebook 3)

**Runtime:** ~15 minutes

In [1]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("06_dspy/10_adapters_evaluation.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  06_dspy/10_adapters_evaluation.ipynb
Task:      Adapters + evaluation metrics
Calls:     ~25

With GPT-4o:       $0.25 USD
With GPT-4o-mini:  $0.03 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 1. Setup

In [2]:
import logging

import dspy

from src.config import get_dspy_lm, print_config
from src.datasets import generate_qa_pairs

logging.getLogger("dspy.evaluate").setLevel(logging.WARNING)  # keep INFO logs out of outputs

print_config()

lm = get_dspy_lm()
dspy.configure(lm=lm)

# Small devset reused across the evaluation demos (same synthetic loader as Notebooks 2 & 9)
data = generate_qa_pairs(12)
devset = [
    dspy.Example(question=d["question"], answer=d["expected"]).with_inputs("question") for d in data
]
print(f"\n✅ DSPy configured with: {lm.model} | devset: {len(devset)} examples")

LLM Libraries — Configuration
USE_OLLAMA:        False
USE_SMALL_MODEL:   False
OLLAMA_BASE_URL:   http://localhost:11434
LLM_PROVIDER:      openai
  openai model:    gpt-4o
  anthropic model: claude-opus-4-6
  gemini model:    gemini-3.6-flash
  groq model:      openai/gpt-oss-120b
SAMPLE_SIZE:       50
DSPY_TRIALS:       10

✅ DSPy configured with: openai/gpt-4o | devset: 12 examples


## Part I — Adapters

## 2. What Is an Adapter?

Every DSPy call travels through an **Adapter**: it renders your signature (inputs +
instructions + output-field spec) into chat messages, sends them to the LM, then parses the
raw text back into the signature's typed output fields. The same `Predict("question -> answer")`
code can therefore speak *delimiter-marked chat*, *XML*, or *JSON* to the model — you swap the
adapter, not your program.

```mermaid
flowchart LR
    A["your dspy.Module"] --> B["Adapter\nformat + parse"]
    B --> C["Language Model"]
    C --> B
    B --> D["Prediction\ntyped output fields"]
    style B fill:#2d6a4f
```

**Configuration, two scopes:**
- Global: `dspy.configure(lm=lm, adapter=dspy.JSONAdapter())` — every call uses it.
- Scoped: `with dspy.context(adapter=dspy.XMLAdapter()): ...` — one block only (used below).

If no adapter is configured, DSPy resolves `ChatAdapter` at call time — which is why you have
been using adapters since Notebook 1 without noticing.

## 3. `Adapter` — The Base Class

**What it is:** the abstract interface all adapters implement.

**Contract:** `__call__(lm, lm_kwargs, signature, demos, inputs) -> list[dict]` — build the
messages, invoke the LM, parse completions into dictionaries keyed by the signature's output
fields. Concrete adapters differ only in *how they format* (the prompt) and *how they parse*
(the reply).

**Key params (inherited by all adapters):**
- `callbacks` — `dspy.BaseCallback` hooks for observability (tracing, logging)
- `use_native_function_calling` — emit the output schema as native tool/function definitions
  instead of text instructions (model-API support required)
- `native_response_types` — custom `dspy.Type` subclasses for native structured responses
- `parallel_tool_calls` — allow multiple tool calls per turn

**When to use:** subclass it only when you need a custom wire format (e.g. a company-internal
API with its own templating); for everything else the four concrete adapters below suffice.

## 4. `ChatAdapter` — The Default (Delimiter Markup)

**What it is:** the default adapter for most chat models. Fields are separated by
`[[ ## field_name ## ]]` marker lines in the message text — unambiguous for the model, easy to
eyeball in logs.

**How it works:** renders inputs under `[[ ## question ## ]]`, appends an output section
listing each expected field, then parses the reply by splitting on the same markers.
`use_json_adapter_fallback=True` (default) means: if marker parsing ever fails, retry through
`JSONAdapter` automatically — free robustness.

**When to use:** default for OpenAI, Anthropic, Gemini, Groq, Ollama chat models. **Avoid
when** the model is fine-tuned for XML/JSON specifically, or is a reasoning model (see
`TwoStepAdapter`). The demo below shows the exact prompt it sends.

In [3]:
predictor = dspy.Predict("question -> answer")
pred = predictor(question="What is the capital of Norway?")

# lm.history holds every raw LM request — inspect the exact rendered prompt
user_msg = next(m["content"] for m in lm.history[-1]["messages"] if m["role"] == "user")
print("=== Prompt ChatAdapter sent to the LM ===")
print(user_msg)
print("=" * 45)
print(f"Parsed back into fields: answer={pred.answer!r}")

=== Prompt ChatAdapter sent to the LM ===
[[ ## question ## ]]
What is the capital of Norway?

Respond with the corresponding output fields, starting with the field `[[ ## answer ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.
Parsed back into fields: answer='Oslo'


## 5. `XMLAdapter` — Tag-Based Markup

**What it is:** same contract, but fields are rendered as XML tags —
`<question>...</question>` in, `<answer>...</answer>` out.

**How it works:** identical pipeline to ChatAdapter, different wire format and parser
(element text → field value). Also carries the JSONAdapter fallback.

**When to use:** models with strong XML fine-tuning — historically Claude models and several
open-weight models score reliably with tag markup. If your model's docs mention XML prompting,
benchmark this adapter against ChatAdapter. **Avoid when** the model wasn't trained on XML —
`ChatAdapter` is usually equal or better, and JSON beats both when native calling is available.

In [4]:
predictor = dspy.Predict("question -> answer")

with dspy.context(adapter=dspy.XMLAdapter()):
    pred = predictor(question="What is the capital of Sweden?")

user_msg = next(m["content"] for m in lm.history[-1]["messages"] if m["role"] == "user")
print("=== Prompt XMLAdapter sent to the LM ===")
print(user_msg)
print("=" * 45)
print(f"Parsed back into fields: answer={pred.answer!r}")

=== Prompt XMLAdapter sent to the LM ===
<question>
What is the capital of Sweden?
</question>

Respond with the corresponding output fields wrapped in XML tags `<answer>`.
Parsed back into fields: answer='Stockholm'


## 6. `JSONAdapter` — Schema-Native Structured Output

**What it is:** renders the output fields as a **JSON schema** in the prompt and
parses the reply as JSON.

**How it works:** two modes. Text mode: instruct the model to reply with a JSON object and
parse/validate it against the signature. Native mode (`use_native_function_calling=True`, the
default): the schema is sent via the API's **structured-output / function-calling mechanism**
(e.g. OpenAI `response_format`) — the strongest structural guarantee the provider offers, with
server-side validation.

**When to use:** strict structured outputs on providers with native support; function/tool
pipelines; eliminating parse failures entirely. **Avoid when** the model lacks JSON/native
support (small local models) — a failed JSON parse is worse than a forgiving marker parse, and
note that long chain-of-thought mixed into JSON fields can still strain weaker models.

In [5]:
predictor = dspy.Predict("question -> answer")

with dspy.context(adapter=dspy.JSONAdapter()):
    pred = predictor(question="What is the capital of Finland?")

print(f"answer={pred.answer!r}")
print("(prompt asked for a JSON object; native structured outputs used where supported)")

answer='Helsinki'
(prompt asked for a JSON object; native structured outputs used where supported)


## 7. `TwoStepAdapter` — Reasoning Model in Front, Parser Behind

**What it is:** a two-stage adapter built for **reasoning models** (o1/o3, DeepSeek-R1
style) that think at length and are notoriously bad at structured output.

**How it works:**
1. The main LM gets a *natural, simple* prompt and answers however it likes — long reasoning
   included.
2. A second, cheap **extraction LM** receives the main model's raw text and parses just the
   final answer into the signature's fields.

You get reasoning quality *and* structured outputs, decoupled. Cost = 1 main call + 1 cheap
extraction call.

**Key params:** `extraction_model` (any `dspy.LM`; use a small cheap one), plus base-Adapter
kwargs.

**When to use:** reasoning-first models, or any model whose prose doesn't parse reliably.
**Avoid when** the main model already follows structured output well — the extra call is pure
overhead then.

In [6]:
# Main LM stays as configured (gpt-4o here); the extractor is a cheap mini model
extractor = dspy.LM("openai/gpt-4o-mini")
adapter = dspy.TwoStepAdapter(extraction_model=extractor)

with dspy.context(adapter=adapter):
    cot = dspy.ChainOfThought("question -> answer")
    pred = cot(question="What is 17 * 23, and which planet is closest to the Sun?")

print(f"Structured answer: {pred.answer!r}")
print("(main LM reasoned freely; gpt-4o-mini extracted the answer field)")

Structured answer: '17 * 23 is 391, and the planet closest to the Sun is Mercury.'
(main LM reasoned freely; gpt-4o-mini extracted the answer field)


## Part II — Evaluation

## 8. The Metric Contract

Everything in DSPy that needs to judge quality — `Evaluate` (below), every optimizer
in Notebook 9, assertions in Notebook 3 — consumes the **same metric function**:

```python
def metric(example, prediction, trace=None) -> float | bool
```

- `example` — a `dspy.Example` (gold inputs/outputs)
- `prediction` — the program's `dspy.Prediction`
- `trace` — set during optimization; LLM-judge metrics use it to return a pass/fail bool
  instead of a raw score

```mermaid
flowchart LR
    D["devset examples"] --> P["your program"]
    P --> M["metric\n(example, prediction, trace)"]
    M --> S["per-example scores"]
    S --> A["EvaluationResult.score\n(0-100 aggregate)"]
    style M fill:#2d6a4f
```

Metrics come in two families: **string/programmatic** (free, deterministic — §9) and
**LLM-judge** (costs calls, handles semantics — §10–11). Choose per task, not per taste:
exact answers want string metrics; open-ended answers want judges.

## 9. `answer_exact_match` / `answer_passage_match` — Free String Metrics

**`answer_exact_match`** — exact string match (after normalization) between
`pred.answer` and `example.answer`; with `frac < 1.0` it accepts answers whose token-F1 against
the reference reaches the threshold, and `example.answer` may be a *list* of acceptable
references. Free, deterministic, no LM calls.

**`answer_passage_match`** — the retrieval metric: True if any passage in `pred.context`
contains any reference answer. Built for RAG pipelines (Notebook 3) where the prediction
carries retrieved evidence.

**When to use:** trivia/QA with short canonical answers, and RAG faithfulness checks.
**Avoid when** paraphrases are acceptable — these punish valid rewordings (see `SemanticF1`).

In [7]:
from dspy.evaluate import answer_exact_match, answer_passage_match

# --- exact match: canonical short answers ------------------------------- #
good = dspy.Prediction(answer="Paris")
bad = dspy.Prediction(answer="Lyon")

print("EM 'Paris' vs 'Paris':", answer_exact_match(dspy.Example(answer="Paris"), good))
print("EM 'Paris' vs 'Lyon': ", answer_exact_match(dspy.Example(answer="Paris"), bad))

# F1-thresholded match: longer answers that CONTAIN the reference can pass at frac < 1.0
almost = dspy.Prediction(answer="Paris, France")
score = answer_exact_match(dspy.Example(answer="Paris"), almost, frac=0.5)
print("F1>=0.5 'Paris, France' vs 'Paris':", score)

# --- passage match: did the retrieved context contain the answer? ------- #
rag_pred = dspy.Prediction(
    answer="It combines retrieval with generation.",
    context=[
        "DSPy is a framework for programming language models.",
        "RAG combines retrieval with generation to ground answers in documents.",
    ],
)
print(
    "Passage match for 'combines retrieval with generation':",
    answer_passage_match(dspy.Example(answer="combines retrieval with generation"), rag_pred),
)
print(
    "Passage match for 'invented in 1998':",
    answer_passage_match(dspy.Example(answer="invented in 1998"), rag_pred),
)

EM 'Paris' vs 'Paris': True
EM 'Paris' vs 'Lyon':  False
F1>=0.5 'Paris, France' vs 'Paris': True
Passage match for 'combines retrieval with generation': True
Passage match for 'invented in 1998': False


## 10. `SemanticF1` — LLM-Judged Precision/Recall

**What it is:** an LLM-based metric that decomposes prediction and gold into semantic
content units, then computes **precision** (how much of the prediction is supported by the
gold) and **recall** (how much of the gold the prediction covers) and returns their F1.

**How it works:** internally a `dspy.ChainOfThought` module asks the LM to enumerate key ideas
on both sides and score overlap; the F1 lands on the returned `Prediction.score`. With
`trace` set (during optimization) it instead returns whether F1 ≥ `threshold` (default 0.66).
`decompositional=True` switches to a finer-grained recall/precision variant.

**Field names matter:** it reads `example.question`, `example.response`, `pred.response`.

**Budget:** ~1–2 calls per judged pair.

**When to use:** open-ended generation where paraphrase is fine but *content coverage* matters
(summaries, explanations, RAG answers). **Avoid when** a string metric already decides
correctly — judges add cost and variance.

In [8]:
from dspy.evaluate import SemanticF1

metric = SemanticF1()

example = dspy.Example(
    question="What does RAG do?",
    response="RAG combines retrieval with generation to ground answers in documents.",
).with_inputs("question")
pred = dspy.Prediction(
    response="Retrieval-augmented generation grounds LLM answers in retrieved documents."
)

result = metric(example, pred)
print(f"SemanticF1: {result.score:.2f}")
print("(exact match would score 0 here — the wording is completely different)")

SemanticF1: 1.00
(exact match would score 0 here — the wording is completely different)


## 11. `CompleteAndGrounded` — Coverage × Faithfulness Judge

**What it is:** an LLM judge that scores RAG-style answers on two axes at once:
- **Completeness** — fraction of the gold response's key ideas covered by the prediction
- **Groundedness** — fraction of the prediction's check-worthy claims supported by the
  retrieved `pred.context`

The final score is the F1 of the two. Like `SemanticF1`, it reads `example.question` /
`example.response` / `pred.response` (+ `pred.context`), and honors `threshold` (default 0.66)
when a trace is present.

**Budget:** ~2 calls per judged pair (one per axis).

**When to use:** evaluating RAG pipelines (with Notebook 3) where an answer must both *cover*
the gold and *stay faithful to retrieved evidence*. **Avoid when** there is no context field —
use `SemanticF1` instead.

In [9]:
from dspy.evaluate import CompleteAndGrounded

metric = CompleteAndGrounded()

example = dspy.Example(
    question="What is DSPy?",
    response="DSPy is a framework for programming language models.",
).with_inputs("question")
pred = dspy.Prediction(
    response="DSPy is a Python framework for programming language models.",
    context=["DSPy is a framework for programming language models."],
)

result = metric(example, pred)
print(f"CompleteAndGrounded: {result.score:.2f}")
print("(high: the answer covers the gold AND every claim is supported by the context)")

CompleteAndGrounded: 0.67
(high: the answer covers the gold AND every claim is supported by the context)


## 12. `Evaluate` + `EvaluationResult` — Devset Scoring at Scale

**`Evaluate`** — the evaluation engine the notebooks have been using under the hood:
runs your program over a devset **concurrently** (`num_threads`), applies the metric to every
(example, prediction) pair, and aggregates.

**Key params:** `devset`, `metric`, `num_threads`, `display_progress` / `display_table`,
`max_errors`.

**`EvaluationResult`** — what a run returns: `.score` (aggregate, 0–100) and `.results`, a list
of `(example, prediction, score)` tuples for per-example inspection.

**When to use:** always, before trusting any program — and re-run it after every optimization
(Notebook 9 optimizers literally consume these scores). **Avoid when** the devset is tiny and
noisy — no metric can fix 5 examples.

In [10]:
from dspy.evaluate import Evaluate


def word_overlap(example, prediction, trace=None):
    expected = set(example.answer.lower().split())
    predicted = set(prediction.answer.lower().split())
    return 1.0 if expected and len(expected & predicted) / len(expected) >= 0.5 else 0.0


cot = dspy.ChainOfThought("question -> answer")

evaluator = Evaluate(devset=devset, metric=word_overlap, num_threads=4, display_progress=False)
result = evaluator(cot)

print(f"Aggregate score: {result.score:.2f}  ({len(result.results)} examples)")
print("\nPer-example scores:")
for example, prediction, score in result.results[:4]:
    print(f"  {score:.0f}  {example.question:<28} -> {prediction.answer[:50]}")

Aggregate score: 41.67  (12 examples)

Per-example scores:
  0  What is DSPy?                -> There is no widely recognized meaning for "DSPy" a
  0  What is Vector DB?           -> Vector DB is a database optimized for storing and 
  1  What is Fine-tuning?         -> Fine-tuning is the process of taking a pre-trained
  0  What is DSPy?                -> There is no widely recognized meaning for "DSPy" a


## Cheat-Sheets

**Adapters**

| Adapter | Wire format | Native calling | Reach for it when… |
|---|---|---|---|
| `ChatAdapter` (default) | `[[ ## field ## ]]` markers | optional | any standard chat model |
| `XMLAdapter` | `<field>` tags | optional | XML-fine-tuned models |
| `JSONAdapter` | JSON schema | **on by default** | strict structured outputs, function-calling APIs |
| `TwoStepAdapter` | natural prompt + extractor LM | — | reasoning models that can't structure output |

**Evaluation**

| Tool | Judge | Cost | Reach for it when… |
|---|---|---|---|
| `answer_exact_match` | string | free | canonical short answers |
| `answer_passage_match` | string | free | RAG: answer present in retrieved context |
| `SemanticF1` | LLM | ~1–2 calls/pair | paraphrase-tolerant coverage scoring |
| `CompleteAndGrounded` | LLM | ~2 calls/pair | RAG answers must cover gold *and* stay grounded |
| `Evaluate` / `EvaluationResult` | harness | devset-sized | every program, before and after optimization |

**Where it all connects:** adapters decide what the LM sees; metrics decide what counts as
correct; optimizers (Notebook 9) close the loop. Next: **Notebook 3** (RAG — where
`answer_passage_match` and `CompleteAndGrounded` earn their keep).